# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [ ]:
import os
from huggingface_hub import login

# login(token="your_huggingface_token_here")  # Replace with your actual Hugging Face token
login(token=os.environ["HF_TOKEN"])


In [ ]:
!uv sync
!uv pip install "unsloth[colab-new]" trl transformers peft accelerate bitsandbytes datasets
!uv add datasets torch unsloth

In [ ]:
!uv run generate_synthetic_cases.py --n-train 160 --n-val   30 --output-dir synthetic_cases

In [ ]:
!uv run build_sft_dataset.py --orig-dir ../ --synth-dir synthetic_cases/ --out-dir   dataset/

In [ ]:
!uv run train_ddi_sft.py --merge --push-hub amank-root/ddi-1.5b-lora

In [ ]:
!uv run train_ddi_sft.py --model Qwen/Qwen3-8B --train-file dataset/ddi_train.jsonl --val-file   dataset/ddi_val.jsonl --output-dir ddi_lora_adapter --epochs 3 --batch-size 2 --grad-accum 4 --lr 2e-4 --lora-r 16 --max-seq-len 1024

In [ ]:
!uv run train_ddi_sft.py --model Qwen/Qwen3-8B --train-file dataset/ddi_train.jsonl --val-file   dataset/ddi_val.jsonl --output-dir ddi_lora_adapter --epochs 3 --batch-size 1 --grad-accum 8 --lr 2e-4 --lora-r 8 --max-seq-len 512

In [ ]:
!uv run upload_to_hub.py --mode adapter --repo amank-root/ddi-1.5b-lora
# !uv run upload_to_hub.py --mode merged --repo amank-root/ddi-1.5b-merged

In [ ]:
!uv run eval_ddi_model.py --model-dir  ddi_lora_adapter --base-model Qwen/Qwen2.5-1.5B-Instruct --val-file   dataset/ddi_val.jsonl  --mode offline

In [ ]:
!uv run eval_ddi_model.py --model-dir  ddi_lora_adapter --base-model Qwen/Qwen2.5-1.5B-Instruct  --openenv-dir ../  --mode online  --episodes 12